In [1]:
import pandas as pd 

a=pd.read_csv(r"D:\CODING\PYTHON\NLP\sample.csv")
a


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,119237,105834,True,Wed Oct 11 06:55:44 +0000 2017,@AppleSupport causing the reply to be disregar...,119236,NaN
1,119238,ChaseSupport,False,Wed Oct 11 13:25:49 +0000 2017,@105835 Your business means a lot to us. Pleas...,NaN,119239.0
2,119239,105835,True,Wed Oct 11 13:00:09 +0000 2017,@76328 I really hope you all change but I'm su...,119238,NaN
3,119240,VirginTrains,False,Tue Oct 10 15:16:08 +0000 2017,@105836 LiveChat is online at the moment - htt...,119241,119242.0
4,119241,105836,True,Tue Oct 10 15:17:21 +0000 2017,@VirginTrains see attached error message. I've...,119243,119240.0
...,...,...,...,...,...,...,...
88,119330,105859,True,Wed Oct 11 13:50:42 +0000 2017,@105860 I wish Amazon had an option of where I...,119329,119331.0
89,119331,105860,True,Wed Oct 11 13:47:14 +0000 2017,They reschedule my shit for tomorrow https://t...,119330,NaN
90,119332,Tesco,False,Wed Oct 11 13:34:06 +0000 2017,"@105861 Hey Sara, sorry to hear of the issues ...",119333,119334.0
91,119333,105861,True,Wed Oct 11 14:05:18 +0000 2017,@Tesco bit of both - finding the layout cumber...,"119335,119336",119332.0


In [3]:
a.keys()

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

In [4]:
a=a.drop(columns=['tweet_id', 'author_id', 'created_at', 
       'response_tweet_id', 'in_response_to_tweet_id'])

In [5]:
a

,inbound,text
0,True,@AppleSupport causing the reply to be disregar...
1,False,@105835 Your business means a lot to us. Pleas...
2,True,@76328 I really hope you all change but I'm su...
3,False,@105836 LiveChat is online at the moment - htt...
4,True,@VirginTrains see attached error message. I've...
...,...,...
88,True,@105860 I wish Amazon had an option of where I...
89,True,They reschedule my shit for tomorrow https://t...
90,False,"@105861 Hey Sara, sorry to hear of the issues ..."
91,True,@Tesco bit of both - finding the layout cumber...


In [7]:
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

pu=PorterStemmer()

def remove(text):
    text=text.lower()
    text=word_tokenize(text)

    k=[]
    for i in text:
        if i.isalnum():
            k.append(i)
    text=k[:]
    k.clear()

    for i in text:
        if i not in string.punctuation and stopwords.words("english"):
            k.append(i)

    text=k[:]
    k.clear()

    for i in text:
        k.append(pu.stem(i))

    return " ".join(k)






In [8]:
a["text"]=a["text"].apply(remove)
a["text"]

0     applesupport caus the repli to be disregard an...
1     105835 your busi mean a lot to us pleas dm you...
2     76328 i realli hope you all chang but i sure y...
3     105836 livechat is onlin at the moment http or...
4     virgintrain see attach error messag i tri leav...
                            ...                        
88    105860 i wish amazon had an option of where i ...
89             they reschedul my shit for tomorrow http
90    105861 hey sara sorri to hear of the issu you ...
91    tesco bit of both find the layout cumbersom an...
92    105861 if that doe help pleas dm your full nam...
Name: text, Length: 93, dtype: object

In [10]:
import emoji

def remove_emoji(text):
    return emoji.demojize(text)

a["text"]=a["text"].apply(remove_emoji)

In [11]:
from sklearn.preprocessing import LabelEncoder

la=LabelEncoder()
a["inbound"]=la.fit_transform(a["inbound"])

In [18]:
a

,inbound,text
0,1,applesupport caus the repli to be disregard an...
1,0,105835 your busi mean a lot to us pleas dm you...
2,1,76328 i realli hope you all chang but i sure y...
3,0,105836 livechat is onlin at the moment http or...
4,1,virgintrain see attach error messag i tri leav...
...,...,...
88,1,105860 i wish amazon had an option of where i ...
89,1,they reschedul my shit for tomorrow http
90,0,105861 hey sara sorri to hear of the issu you ...
91,1,tesco bit of both find the layout cumbersom an...


In [25]:
x=a.iloc[:,-1]
y=a["inbound"]
x

0     applesupport caus the repli to be disregard an...
1     105835 your busi mean a lot to us pleas dm you...
2     76328 i realli hope you all chang but i sure y...
3     105836 livechat is onlin at the moment http or...
4     virgintrain see attach error messag i tri leav...
                            ...                        
88    105860 i wish amazon had an option of where i ...
89             they reschedul my shit for tomorrow http
90    105861 hey sara sorri to hear of the issu you ...
91    tesco bit of both find the layout cumbersom an...
92    105861 if that doe help pleas dm your full nam...
Name: text, Length: 93, dtype: object

In [26]:
y

0     1
1     0
2     1
3     0
4     1
     ..
88    1
89    1
90    0
91    1
92    0
Name: inbound, Length: 93, dtype: int64

In [27]:
from sklearn.model_selection import train_test_split 
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer()
x_train=tf.fit_transform(x_train).toarray()
x_test=tf.transform(x_test).toarray()

In [29]:
x_train.shape

(74, 494)

In [40]:
y_train.shape

(74,)

In [30]:
x_test.shape

(19, 494)

In [31]:
from keras import Sequential
from keras.layers import Dense,SimpleRNN

In [61]:
model=Sequential()
# model.add(Dense(34,input_dim=19,activation="sigmoid"))
model.add(SimpleRNN(20,input_shape=(19,1),return_sequences=False))
model.add(Dense(30,activation="relu"))

# model.add(SimpleRNN(30))
model.add(Dense(1,activation="tanh"))

In [62]:
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_15 (SimpleRNN)       │ (None, 20)             │           440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 30)             │           630 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,101 (4.30 KB)

 Trainable params: 1,101 (4.30 KB)

 Non-trainable params: 0 (0.00 B)

In [63]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [64]:
model.fit(x_train,y_train,epochs=10,validation_data=(x_test,y_test))

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 170ms/step - accuracy: 0.4865 - loss: 3.4774 - val_accuracy: 0.4211 - val_loss: 3.4538
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.4865 - loss: 2.6360 - val_accuracy: 0.4211 - val_loss: 1.2487
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4865 - loss: 2.3133 - val_accuracy: 0.4211 - val_loss: 2.1429
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.4730 - loss: 2.2809 - val_accuracy: 0.4211 - val_loss: 2.5216
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.4865 - loss: 2.0409 - val_accuracy: 0.4211 - val_loss: 1.7254
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5000 - loss: 1.1116 - val_accuracy: 0.4211 - val_loss: 1.3134
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.4730 - loss: 1.7405 - val_accuracy: 0.4211 - val_loss: 1.0423
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.5000 - loss: 0.9851 - val_accuracy: 0.4737 - val_loss: 0.8952

In [42]:
from sklearn.linear_model import LogisticRegression

lo=LogisticRegression()
lo.fit(x_train,y_train)
lo.score(x_test,y_test)

0.7894736842105263